## Comparação exploratória com benchmark da OIT

Este notebook realiza apenas as análises necessárias para a subseção 4.2 (Comparação com o benchmark da OIT) e para apoiar a conclusão do TCC, avaliando a coerência externa exploratória do índice de exposição ocupacional (`indice_0_1`) em relação ao benchmark internacional (`mean_ilo`).

### 1. Importação e configuração

Importamos as bibliotecas necessárias e definimos caminhos de entrada e saída.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk")

def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for c in candidates:
        if (c / "dados" / "indice_ocupacao_cbo.csv").exists():
            return c
    raise FileNotFoundError(
        "Não foi possível localizar `dados/indice_ocupacao_cbo.csv`. "
        "Verifique se o notebook foi executado a partir do diretório correto."
    )

BASE_DIR = find_project_root()
INPUT_CBO = BASE_DIR / "dados" / "indice_ocupacao_cbo.csv"
INPUT_OIT = BASE_DIR / "dados" / "indice_ocupacoes_oit.csv"
OUTPUT_DIR_TABELAS = BASE_DIR / "resultados" / "tabelas"
OUTPUT_DIR_GRAFICOS = BASE_DIR / "resultados" / "graficos"
OUTPUT_DIR = BASE_DIR / "resultados"  # compatibilidade
OUTPUT_DIR_TABELAS.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_GRAFICOS.mkdir(parents=True, exist_ok=True)

### 2. Leitura, validação e consolidação das bases

Lemos os dois arquivos CSV, validamos colunas e número de linhas, e unimos os dados assumindo que as 30 ocupações já estão mapeadas linha a linha entre CBO e OIT.

In [3]:
REQUIRED_CBO = ["ocupacao", "indice_0_1"]
REQUIRED_OIT = ["ocupacao_equivalente_ILO", "codigo", "gradiente_ilo", "mean_ilo"]

def load_and_validate_cbo(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8")
    df.columns = df.columns.str.strip()
    missing = [c for c in REQUIRED_CBO if c not in df.columns]
    if missing:
        raise ValueError(f"Colunas ausentes em {path.name}: {missing}")
    return df

def load_and_validate_oit(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8")
    df.columns = df.columns.str.strip()
    missing = [c for c in REQUIRED_OIT if c not in df.columns]
    if missing:
        raise ValueError(f"Colunas ausentes em {path.name}: {missing}")
    return df

df_cbo = load_and_validate_cbo(INPUT_CBO)
df_oit = load_and_validate_oit(INPUT_OIT)

print("Dimensões CBO:", df_cbo.shape)
print("Dimensões OIT:", df_oit.shape)

if len(df_cbo) != len(df_oit):
    raise ValueError("Número de linhas diferente entre CBO e OIT; não é possível unir por ordem.")

# Resetar índices para garantir alinhamento por posição
df_cbo = df_cbo.reset_index(drop=True)
df_oit = df_oit.reset_index(drop=True)

# Consolidação linha a linha (apenas as 6 colunas solicitadas)
df_merge = pd.concat([df_cbo[["ocupacao", "indice_0_1"]], df_oit], axis=1)

# Garantir índices numéricos (CBO usa vírgula como decimal)
df_merge["indice_0_1"] = (
    df_merge["indice_0_1"].astype(str).str.replace(",", ".", regex=False)
)
df_merge["indice_0_1"] = pd.to_numeric(df_merge["indice_0_1"], errors="coerce")
df_merge["mean_ilo"] = pd.to_numeric(df_merge["mean_ilo"], errors="coerce")

# DataFrame consolidado: ocupacao, indice_0_1, ocupacao_equivalente_ILO, codigo, gradiente_ilo, mean_ilo
df = df_merge.dropna(subset=["indice_0_1", "mean_ilo"]).copy()
print("Linhas consolidadas:", len(df))
display(df.head())

ValueError: Colunas ausentes em indice_ocupacao_cbo.csv: ['n_atividades', 'media_scores_atividades']

_Interpretação breve_: Esta etapa garante que as duas fontes estejam consistentes em número de ocupações e que possam ser unidas pela ordem das linhas, respeitando o mapeamento prévio entre CBO e OIT.

### 3. Ordenamentos

Construímos o ranking das ocupações pelo `indice_0_1` e pelo `mean_ilo` (maior valor = posição mais alta), e a diferença de posições entre os dois ordenamentos.

In [4]:
# Rank pelo índice próprio (1 = maior exposição)
df["rank_indice"] = df["indice_0_1"].rank(ascending=False, method="first").astype(int)
# Rank pelo benchmark OIT (1 = maior mean_ilo)
df["rank_oit"] = df["mean_ilo"].rank(ascending=False, method="first").astype(int)
# Diferença de posições entre os dois rankings
df["diff_rank"] = (df["rank_indice"] - df["rank_oit"]).abs()

print("Exemplo: ranking pelo indice_0_1 (primeiras posições)")
display(df.nsmallest(5, "rank_indice")[["ocupacao", "indice_0_1", "rank_indice", "rank_oit", "diff_rank"]])
print("Exemplo: ranking pelo mean_ilo (primeiras posições)")
display(df.nsmallest(5, "rank_oit")[["ocupacao", "mean_ilo", "rank_oit", "rank_indice", "diff_rank"]])
print("Diferença de posições (diff_rank): min = {}, max = {}".format(df["diff_rank"].min(), df["diff_rank"].max()))

NameError: name 'df' is not defined

_Interpretação breve_: A comparação entre ordenamentos permite avaliar coerência externa sem assumir equivalência de escala. Ocupações com `diff_rank` próximo de zero ocupam posições semelhantes nos dois rankings.

### 4. Medidas de associação por ordenamento

A medida principal é a correlação de Spearman (ordenação). Pearson é informativa apenas como complemento. Inclui-se a distribuição das ocupações por gradiente_ilo.

In [5]:
# Correlação de Spearman (principal: associação por ordenação)
spearman_r, spearman_p = stats.spearmanr(df["indice_0_1"], df["mean_ilo"])
print("Correlação de Spearman (indice_0_1 vs mean_ilo): rho = {:.3f}, p = {:.4f}".format(spearman_r, spearman_p))

# Pearson apenas como informação complementar (não evidência principal)
pearson_r, pearson_p = stats.pearsonr(df["indice_0_1"], df["mean_ilo"])
print("Correlação de Pearson (complementar): r = {:.3f}, p = {:.4f}".format(pearson_r, pearson_p))

# Distribuição das ocupações por gradiente_ilo
dist_gradiente = df["gradiente_ilo"].value_counts().sort_index()
print("\nDistribuição das ocupações por gradiente_ilo:")
display(dist_gradiente.to_frame("n_ocupacoes"))

NameError: name 'df' is not defined

_Interpretação breve_: As estatísticas descritivas e correlações quantificam a associação geral entre o índice desenvolvido e o benchmark da OIT, sem assumir equivalência direta de escala, enquanto as diferenças simples indicam discrepâncias de magnitude ocupação a ocupação.

### 5. Convergências e divergências entre ordenamentos

Identificamos ocupações com maior convergência (menor diferença de posição) e maior divergência (maior diferença de posição) entre os dois rankings. Tabela final com as 7 colunas solicitadas.

In [6]:
# Convergência = menor diferença de posição entre os dois rankings
top_convergentes = df.sort_values("diff_rank").head(5)
print("5 ocupações com maior convergência entre os dois ordenamentos (menor diff_rank):")
display(top_convergentes[["ocupacao", "indice_0_1", "rank_indice", "mean_ilo", "gradiente_ilo", "rank_oit", "diff_rank"]])

# Divergência = maior diferença de posição entre os dois rankings
top_divergentes = df.sort_values("diff_rank", ascending=False).head(5)
print("5 ocupações com maior divergência entre os dois ordenamentos (maior diff_rank):")
display(top_divergentes[["ocupacao", "indice_0_1", "rank_indice", "mean_ilo", "gradiente_ilo", "rank_oit", "diff_rank"]])

# Tabela final: ocupacao, indice_0_1, rank no índice próprio, mean_ilo, gradiente_ilo, rank na OIT, diferença de rank
tabela_final = df[[
    "ocupacao",
    "indice_0_1",
    "rank_indice",
    "mean_ilo",
    "gradiente_ilo",
    "rank_oit",
    "diff_rank",
]].copy()
tabela_final = tabela_final.rename(columns={"rank_indice": "rank_no_indice", "rank_oit": "rank_na_OIT", "diff_rank": "diferenca_de_rank"})
tabela_final = tabela_final.sort_values("rank_no_indice")
display(tabela_final)
tabela_final.to_csv(OUTPUT_DIR_TABELAS / "comparacao_cbo_oit_tabela_final.csv", index=False)

NameError: name 'df' is not defined

_Interpretação breve_: As listas de maior convergência e maior divergência ajudam a identificar quais ocupações reforçam a coerência externa do índice e quais demandam análise qualitativa mais cuidadosa na discussão do TCC.

### 6. Análise por gradiente da OIT

Avaliamos como o índice desenvolvido se distribui por categorias de `gradiente_ilo`, explorando a coerência entre níveis de exposição definidos pela OIT e as médias do índice calculado para a CBO.

In [7]:
# Contagem de ocupações por gradiente
freq_gradiente = df["gradiente_ilo"].value_counts().sort_index()
print("Frequência de ocupações por gradiente_ilo:")
display(freq_gradiente.to_frame("n_ocupacoes"))

# Média e mediana de indice_0_1 dentro de cada gradiente
stats_por_gradiente = df.groupby("gradiente_ilo")["indice_0_1"].agg([
    "count",
    "mean",
    "median",
]).reset_index()

display(stats_por_gradiente)
stats_por_gradiente.to_csv(OUTPUT_DIR_TABELAS / "indice_por_gradiente_oit.csv", index=False)

NameError: name 'df' is not defined

_Interpretação breve_: Se gradientes mais altos da OIT tendem a apresentar valores médios maiores de `indice_0_1`, isso sugere alinhamento substantivo entre os dois referenciais de exposição, ainda que em escalas distintas.

### 7. Gráficos

Produzimos apenas os gráficos essenciais para a subseção 4.2: dispersão entre os índices, comparação por ocupação, maiores divergências e distribuição por gradiente da OIT.

In [ ]:
# Gráfico 1: scatter entre indice_0_1 e mean_ilo com linha de tendência
# Observação: não interpretar como comparação direta de magnitude; apenas visualização do alinhamento geral.
plt.figure(figsize=(8, 6))
sns.regplot(
    data=df,
    x="indice_0_1",
    y="mean_ilo",
    scatter_kws={"alpha": 0.8},
    line_kws={"color": "red"},
)
plt.title("Relação entre índice de exposição (CBO) e benchmark OIT")
plt.xlabel("Índice de exposição desenvolvido (indice_0_1)")
plt.ylabel("Benchmark OIT (mean_ilo)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR_GRAFICOS / "scatter_indice_vs_mean_ilo.png", dpi=300)
plt.show()

In [ ]:
# Gráfico 2: slope chart comparando posição (rank) no índice próprio vs na OIT
df_slope = df.copy()
df_slope = df_slope.sort_values("rank_indice").reset_index(drop=True)
df_slope["ordem"] = df_slope.index  # ordem no eixo x

plt.figure(figsize=(10, 8))
for _, row in df_slope.iterrows():
    plt.plot(
        [0, 1],
        [row["rank_indice"], row["rank_oit"]],
        color="gray",
        alpha=0.6,
    )

plt.scatter(np.zeros(len(df_slope)), df_slope["rank_indice"], label="Rank no índice (CBO)")
plt.scatter(np.ones(len(df_slope)), df_slope["rank_oit"], label="Rank na OIT")

plt.xticks([0, 1], ["Índice CBO", "Benchmark OIT"])
plt.ylabel("Posição (rank)")
plt.title("Comparação de ranking por ocupação: posição no índice vs OIT")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR_GRAFICOS / "slope_indice_vs_mean_ilo.png", dpi=300)
plt.show()

In [ ]:
# Gráfico 3: barplot horizontal das maiores divergências de ranking entre os dois ordenamentos
# Exporta dados + um XLSX com gráfico editável no Excel

top_div_plot = df.sort_values("diff_rank", ascending=False).head(10).copy()

# ===== Export para Excel (dados do gráfico) =====
output_dir_tabelas = globals().get("OUTPUT_DIR_TABELAS", Path.cwd())
output_dir_tabelas = Path(output_dir_tabelas)
output_dir_tabelas.mkdir(parents=True, exist_ok=True)

excel_df = top_div_plot[["ocupacao", "diff_rank"]].copy()

csv_path = output_dir_tabelas / "bar_maiores_divergencias_dados.csv"
excel_df.to_csv(csv_path, index=False, encoding="utf-8")

xlsx_path = output_dir_tabelas / "bar_maiores_divergencias.xlsx"

try:
    from openpyxl import Workbook
    from openpyxl.chart import BarChart, Reference
except Exception as e:
    print(f"[WARN] Nao foi possivel gerar XLSX com chart: {e}")
else:
    wb = Workbook()
    ws = wb.active
    ws.title = "dados"

    ws.append(["Ocupacao", "Diff_rank"])
    for _, r in excel_df.iterrows():
        ws.append([str(r["ocupacao"]), float(r["diff_rank"])])

    n_rows = len(excel_df) + 1  # + header

    cats = Reference(ws, min_col=1, max_col=1, min_row=2, max_row=n_rows)
    data = Reference(ws, min_col=2, max_col=2, min_row=1, max_row=n_rows)

    chart = BarChart()
    chart.type = "bar"
    chart.title = "Maiores divergencias de ranking (Indice vs. OIT)"
    chart.y_axis.title = "Ocupacao (CBO)"
    chart.x_axis.title = "Diferenca de posicao (diff_rank)"

    chart.add_data(data, titles_from_data=True)
    chart.set_categories(cats)

    chart.height = 14
    chart.width = 26

    ws.add_chart(chart, "E2")
    wb.save(xlsx_path)

    print(f"Dados do gráfico (CSV): {csv_path}")
    print(f"Gráfico editável no Excel (XLSX): {xlsx_path}")

# ===== Plot (mantém o gráfico original no notebook) =====
plt.figure(figsize=(10, 8))

sns.barplot(
    data=top_div_plot,
    x="diff_rank",
    y="ocupacao",
)

plt.xlabel("Diferenca de posicao entre rankings (diff_rank)")
plt.ylabel("Ocupacao (CBO)")
plt.title("Ranking da diferenca de posicao (Indice vs. OIT)")
plt.tight_layout()

save_path = OUTPUT_DIR_GRAFICOS / "bar_maiores_divergencias.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Gráfico salvo em: {save_path}")

In [ ]:
# Gráfico 4: barplot com frequência de ocupações por gradiente_ilo

# freq_gradiente é uma Series com índice = gradiente_ilo e valor = contagem
freq_grad_plot = (
    freq_gradiente
    .rename_axis("gradiente_ilo")
    .reset_index(name="n_ocupacoes")
)

plt.figure(figsize=(8, 6))
sns.barplot(
    data=freq_grad_plot,
    x="gradiente_ilo",
    y="n_ocupacoes",
)
plt.xlabel("Gradiente de exposição segundo OIT (gradiente_ilo)")
plt.ylabel("Número de ocupações")
plt.title("Distribuição das ocupações por gradiente de exposição da OIT")
plt.tight_layout()
plt.savefig(OUTPUT_DIR_GRAFICOS / "bar_freq_gradiente_oit.png", dpi=300)
plt.show()

### 8. Saídas finais: tabela, resumo para 4.2 e insights para conclusão

Exportação da tabela consolidada, geração de um resumo textual automático em português acadêmico (base para a subseção 4.2) e bloco de insights úteis para a conclusão do TCC.

In [ ]:
# Tabela consolidada já foi salva em comparacao_cbo_oit_tabela_final.csv na seção 5
# Reexportar com nome explícito para saídas finais
tabela_final.to_csv(OUTPUT_DIR_TABELAS / "comparacao_cbo_oit_consolidado.csv", index=False)
print("Tabela consolidada salva em:", OUTPUT_DIR / "comparacao_cbo_oit_consolidado.csv")

# Resumo textual automático (1–2 parágrafos) para a subseção 4.2
resumo_42 = (
    "A comparação exploratória entre o índice de exposição ocupacional desenvolvido (escala 0–1, baseado em scores de LLMs para atividades da CBO) e o benchmark da OIT (mean_ilo), considerando 30 ocupações mapeadas linha a linha, indicou associação positiva entre os ordenamentos: a correlação de Spearman foi de {:.3f} (p = {:.4f}), sugerindo que ocupações classificadas como mais expostas no índice próprio tendem a figurar também em faixas mais elevadas no indicador internacional. "
    "Não se assume equivalência direta entre as magnitudes de indice_0_1 e mean_ilo, pois as metodologias de construção e agregação diferem; a análise enfatiza, portanto, a coerência do ordenamento e a identificação de convergências e divergências de posição. "
    "Parte das ocupações apresentou posições muito próximas nos dois rankings (maior convergência), ao passo que outras exibiram diferenças de posição mais acentuadas (maior divergência), o que pode ser discutido à luz de particularidades da classificação CBO versus OIT e do desenho do experimento com modelos de linguagem."
).format(spearman_r, spearman_p)

print("\n--- Resumo para a subseção 4.2 (Comparação com o benchmark da OIT) ---\n")
print(resumo_42)

# Salvar resumo em arquivo de texto
with open(OUTPUT_DIR / "resumo_subsecao_42.txt", "w", encoding="utf-8") as f:
    f.write(resumo_42)
print("\nResumo salvo em:", OUTPUT_DIR / "resumo_subsecao_42.txt")

**Insights para a conclusão do TCC**  

- **Coerência externa exploratória:** A comparação com o benchmark da OIT foi tratada como análise de coerência externa exploratória, e não como validação por equivalência numérica. O alinhamento dos ordenamentos (Spearman) e a distribuição das ocupações por gradiente_ilo oferecem indícios de que o índice desenvolvido captura, em parte, padrões semelhantes aos da referência internacional.  

- **Limites da comparação:** Os valores de mean_ilo e indice_0_1 não são diretamente comparáveis em nível absoluto; o mapeamento CBO–OIT é aproximado e as metodologias de construção dos dois indicadores diferem. Divergências de ranking podem refletir diferenças de contexto, de definição ocupacional ou de critérios de exposição, e devem ser discutidas com cautela.  

- **Potencial do índice como teste metodológico:** O exercício de comparação com a OIT ilustra o potencial do índice baseado em LLMs como ferramenta exploratória e como teste metodológico em estudos de exposição ocupacional à IA generativa, sem substituir indicadores consolidados ou validações externas mais robustas.